# Tutorial 1: Querying SAHA Metadata

This notebook shows how to explore the four SAHA metadata tables — `samples`, `donors`, `runs`, and `panels` — using pandas locally or Amazon Athena via SQL.

**No AWS credentials required** for the open-access metadata.

## Setup

In [ ]:
# Install dependencies if needed
# !pip install pandas pyarrow s3fs fsspec

In [ ]:
import pandas as pd

S3_OPTS = {"anon": True}  # open data — no credentials needed
META_BASE = "s3://saha-open-data/metadata"

# Load all four metadata tables
samples = pd.read_parquet(f"{META_BASE}/samples/", storage_options=S3_OPTS)
donors  = pd.read_parquet(f"{META_BASE}/donors/",  storage_options=S3_OPTS)
runs    = pd.read_parquet(f"{META_BASE}/runs/",    storage_options=S3_OPTS)
panels  = pd.read_parquet(f"{META_BASE}/panels/",  storage_options=S3_OPTS)

print(f"samples: {samples.shape}")
print(f"donors:  {donors.shape}")
print(f"runs:    {runs.shape}")
print(f"panels:  {panels.shape}")

## Browse the tables

In [ ]:
samples.head(3)

In [ ]:
donors.head(3)

In [ ]:
panels[["panel_name", "platform", "plex", "assay_type"]]

## Dataset overview

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

qc = samples[samples["qc_pass"] == True]

qc["organ"].value_counts().plot(kind="barh", ax=axes[0], title="Samples by organ")
qc["platform"].value_counts().plot(kind="bar",  ax=axes[1], title="Samples by platform")
qc["condition"].value_counts().plot(kind="pie", ax=axes[2], title="Condition",
                                     autopct="%1.0f%%", startangle=90)

plt.tight_layout()
plt.show()

## Filter samples

In [ ]:
# QC-passing colon samples on CosMx
colon_cosmx = samples.query(
    "organ == 'colon' and platform == 'cosmx' and qc_pass == True"
)
print(f"{len(colon_cosmx)} samples")
colon_cosmx[["sample_id", "run_id", "panel_name", "n_cells", "n_fovs"]].head(10)

In [ ]:
# Samples with large cell counts (>= 20,000 cells)
large = samples[samples["n_cells"] >= 20_000].sort_values("n_cells", ascending=False)
large[["sample_id", "organ", "platform", "n_cells", "panel_plex"]].head(10)

In [ ]:
# Samples grouped by organ + platform: total cells
summary = (
    samples[samples["qc_pass"] == True]
    .groupby(["organ", "platform"])
    .agg(n_samples=("sample_id", "count"), total_cells=("n_cells", "sum"))
    .reset_index()
    .sort_values("total_cells", ascending=False)
)
summary.head(15)

## Join samples → donors

In [ ]:
# Merge on donor_id (registered-tier donors include sex and age_group)
merged = samples.merge(donors[["donor_id", "sex", "age_group"]], on="donor_id", how="left")

# Cell counts by sex
merged.groupby("sex")["n_cells"].sum().plot(kind="bar", title="Total cells by donor sex")
plt.ylabel("Total cells")
plt.show()

In [ ]:
# Multi-organ donors: how many organs per donor?
organs_per_donor = (
    samples[samples["qc_pass"] == True]
    .groupby("donor_id")["organ"]
    .nunique()
    .sort_values(ascending=False)
)
print("Organs per donor (top 10):")
print(organs_per_donor.head(10))

## Join samples → runs

In [ ]:
# Samples with run-level metadata
with_runs = samples.merge(
    runs[["run_id", "run_date", "generation_institution", "instrument_software_version"]],
    on="run_id", how="left",
)
with_runs[["sample_id", "organ", "run_date", "generation_institution"]].head()

## Query panels

In [ ]:
# Panel gene lists (when populated)
for _, row in panels.iterrows():
    gene_count = len(row["gene_list"]) if row["gene_list"] is not None else 0
    print(f"{row['panel_name']:25s}  {row['platform']:8s}  plex={row['plex']:6}  genes={gene_count}")

## Athena SQL (optional)

If you have an AWS account, you can query the same data with SQL using Amazon Athena — no data download required.

In [ ]:
# !pip install pyathena

# from pyathena import connect
# import pandas as pd
#
# conn = connect(
#     s3_staging_dir="s3://your-athena-results-bucket/",
#     region_name="us-east-1",
# )
#
# df = pd.read_sql("""
#     SELECT s.sample_id, s.organ, s.n_cells, d.sex
#     FROM saha.samples s
#     JOIN saha.donors d ON s.donor_id = d.donor_id
#     WHERE s.qc_pass = true AND s.organ = 'colon'
#     ORDER BY s.n_cells DESC
# """, conn)
# df.head()

## Next steps

- **Tutorial 02** — Load and explore a processed CosMx AnnData object
- **Tutorial 03** — Cross-platform comparison of the same tissue
- See [QUICK_START.md](../docs/QUICK_START.md) for more access patterns